### DAY 4 (23/02/26) – Structured Streaming (Basic Simulation)
####🏗️ Architecture & Strategy
Welcome to Day 4! Until now, we have processed data in "batch" mode—reading a static file and transforming it. But in modern eCommerce, data never stops arriving. Users are clicking, viewing, and purchasing 24/7. To capture this, we need **Structured Streaming**.

####Our Strategy:

* **The Landing Zone**: We will create a temporary directory (/tmp/landing_zone/) to act as our data drop-off point, mimicking an AWS S3 bucket or Azure Data Lake container where new logs arrive continuously.

* **Strict Schema Definition**: Unlike batch processing, streams cannot infer schema on the fly. We must strictly define the schema upfront so Spark knows exactly how to parse incoming bytes.

* **Checkpointing**: This is the most critical concept in streaming. We will define a checkpointLocation. Checkpoints store the state of the stream (which files have been processed). If the cluster crashes, the stream restarts exactly where it left off without duplicating data (Exactly-Once Semantics).

* **The Producer/Consumer Simulation**: We will start the stream (the Consumer), and then run a script to drop files into the landing zone (the Producer) to watch the Delta table update in near real-time.

###Setup Landing Zone & Define Schema
First, we clear out any old data from previous runs to ensure a clean simulation. Then, we extract the exact schema from our Day 1 Delta table so our stream knows what to expect.

In [0]:
import shutil

# 1. Define Catalog, Schema, and Target Table
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
target_table = f"{catalog_name}.{schema_name}.streaming_events_bronze"
volume_name = "streaming_assets"

print("🔄 Setting up Unity Catalog context...")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# 2. Create a Unity Catalog Volume (The modern alternative to DBFS /tmp/)
print(f"📦 Creating UC Volume: '{volume_name}' to store streaming files safely...")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

# 3. Define Paths INSIDE the secure Volume
base_volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
landing_zone = f"{base_volume_path}/landing_zone/"
checkpoint_path = f"{base_volume_path}/checkpoints/"

print("🧹 Cleaning up previous simulation directories (if any)...")
dbutils.fs.rm(landing_zone, recurse=True)
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.mkdirs(landing_zone)

# Reset target table for a clean simulation
spark.sql(f"DROP TABLE IF EXISTS {target_table}")

# 4. Enforce Schema for Streaming
# Streaming requires a strict schema. We extract it from our existing batch table.
source_table = f"{schema_name}.events_delta_managed"
events_schema = spark.table(source_table).schema

print(f"✅ Landing Zone ready at: {landing_zone}")
print(f"✅ Checkpoint Path ready at: {checkpoint_path}")
print("✅ Schema loaded successfully.")

####Start the Structured Stream (The Consumer)
We now configure the readStream and writeStream. Notice the .option("maxFilesPerTrigger", 1). In a simulation, this forces Spark to process one file at a time, making it easier to see the micro-batches in action.

In [0]:
import time

# ---------------------------------------------------------
# THE PRODUCER: DROPPING FILES INTO LANDING ZONE (Run First)
# ---------------------------------------------------------
print("🚧 Simulating incoming data traffic...")

# Grab 1,500 rows from our historical table to act as our "new" streaming data
df_sample_data = spark.table(source_table).limit(1500)

# Split the data into 3 chunks of 500 rows and write them to the landing zone
for i in range(1, 4):
    print(f"   ➤ Dropping Payload {i} into landing zone...")
    chunk = df_sample_data.limit(500 * i).subtract(df_sample_data.limit(500 * (i - 1)))
    
    # Write the chunk to the folder the stream will watch
    chunk.coalesce(1).write.format("csv").option("header", "true").mode("append").save(landing_zone)
    time.sleep(1)

print("✅ All payloads delivered. The landing zone is full and ready for the stream!")

Simulate Data Arrival (The Producer)
Now that the stream is running in the background, we will simulate real-world traffic by generating 3 distinct CSV files and dropping them into the landing zone one by one.

In [0]:
# ---------------------------------------------------------
# THE CONSUMER: STARTING THE STREAM (Run Second)
# ---------------------------------------------------------
print("🚀 Initializing Structured Stream (AvailableNow Trigger)...")

# 1. Read Stream
stream_df = (
    spark.readStream
    .schema(events_schema)
    .option("header", "true")
    .csv(landing_zone)
)

# 2. Write Stream with Trigger.AvailableNow
streaming_query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path) 
    .trigger(availableNow=True) # <--- THE CRITICAL FIX
    .queryName("ecommerce_event_stream") 
    .toTable(target_table) 
)

# Because it's AvailableNow, we can tell the notebook to wait until it finishes processing
print("⏳ Stream is actively processing the landing zone. Please wait...")
streaming_query.awaitTermination()

print(f"✅ Stream successfully processed all available files and safely shut down.")
print(f"   Target Table updated: {target_table}")

In [0]:
# ---------------------------------------------------------
# QUERY STREAMING RESULTS
# ---------------------------------------------------------

print(f"📊 Querying target Delta table ({target_table}):")

# Count the records to prove the stream worked
count_df = spark.sql(f"SELECT COUNT(*) as total_streamed_records FROM {target_table}")
display(count_df)

# Show a sample of the data that was streamed
display(spark.sql(f"SELECT * FROM {target_table} LIMIT 5"))

####Key Learnings & Interview Talking Points
If an interviewer asks about your experience with real-time data or Spark Structured Streaming, highlight these points:

* **Micro-Batch Architecture**: "I understand that Spark Structured Streaming operates on a micro-batch architecture. By configuring triggers, I can balance latency requirements with compute efficiency."

* **Checkpointing for Fault Tolerance**: "I never deploy a streaming job without a `checkpointLocation`. This writes offsets and metadata to cloud storage, ensuring exactly-once processing guarantees. If the cluster crashes, it recovers seamlessly without data loss or duplication."

* **Delta Lake as a Streaming Sink**: "Delta Lake is uniquely positioned for streaming because of its ACID transactions. It handles concurrent appends from streams while allowing analysts to query the exact same table simultaneously without reading partial or corrupted files."

* **Explicit Schema Definition**: "Unlike batch processing where `inferSchema` is acceptable, streaming pipelines require strictly enforced schemas to prevent pipeline failures if a malformed payload arrives."